# Pertemuan 13
## Information Flows and Technology

Membangun **Supply Chain Copilot** — AI yang bisa menjawab pertanyaan tentang data rantai pasok beras secara natural.

---
## 1. Dataset Rantai Pasok

In [2]:
import pandas as pd

df = pd.DataFrame({
    'tanggal':  pd.date_range('2024-01-01', periods=12, freq='ME'),
    'provinsi': ['Jawa Barat', 'Jawa Tengah', 'Jawa Timur', 'Lampung',
                 'Sulawesi Sel', 'Jawa Barat', 'Jawa Tengah', 'Jawa Timur',
                 'Lampung', 'Sulawesi Sel', 'Jawa Barat', 'Jawa Tengah'],
    'stok_ton':   [5000, 4800, 4500, 3900, 3200, 4700, 4300, 4100, 3600, 2900, 4400, 4000],
    'harga_rp':   [12000, 12200, 12800, 13200, 13800, 13000, 13500, 14000, 13700, 14200, 13300, 13600],
    'pasokan_ton':[6000, 5500, 5200, 4800, 4000, 5800, 5100, 4900, 4500, 3800, 5400, 4700],
})

df

,tanggal,provinsi,stok_ton,harga_rp,pasokan_ton
0,2024-01-31,Jawa Barat,5000,12000,6000
1,2024-02-29,Jawa Tengah,4800,12200,5500
2,2024-03-31,Jawa Timur,4500,12800,5200
3,2024-04-30,Lampung,3900,13200,4800
4,2024-05-31,Sulawesi Sel,3200,13800,4000
5,2024-06-30,Jawa Barat,4700,13000,5800
6,2024-07-31,Jawa Tengah,4300,13500,5100
7,2024-08-31,Jawa Timur,4100,14000,4900
8,2024-09-30,Lampung,3600,13700,4500
9,2024-10-31,Sulawesi Sel,2900,14200,3800


---
## 2. Supply Chain Copilot

Konsep: **RAG (Retrieval Augmented Generation)** — data nyata disisipkan ke dalam prompt agar AI menjawab berdasarkan fakta.

In [ ]:
from groq import Groq
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

# Ringkasan data sebagai konteks untuk AI
context = (
    "Data rantai pasok beras (Jan–Des 2024):\n"
    + df.to_string(index=False)
    + f"\n\nStatistik:"
    f"\n- Harga tertinggi: Rp {df['harga_rp'].max():,} ({df.loc[df['harga_rp'].idxmax(), 'tanggal'].strftime('%b %Y')})"
    f"\n- Stok terendah: {df['stok_ton'].min():,} ton ({df.loc[df['stok_ton'].idxmin(), 'tanggal'].strftime('%b %Y')})"
    f"\n- Provinsi dengan pasokan terendah: {df.groupby('provinsi')['pasokan_ton'].mean().idxmin()}"
)

def copilot(pertanyaan: str) -> str:
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "Kamu adalah Supply Chain Copilot. Gunakan data berikut untuk menjawab:\n" + context},
            {"role": "user",   "content": pertanyaan}
        ]
    )
    return response.choices[0].message.content

print("Supply Chain Copilot siap!")

Supply Chain Copilot siap!


---
## 3. Tanya Jawab dengan Copilot

In [4]:
print(copilot("Mengapa harga naik?"))

Harga naik karena beberapa kemungkinan faktor, antara lain:

1. **Stok yang menurun**: Pada bulan Oktober 2024, stok beras mencapai titik terendah sebesar 2.900 ton. Ketika stok menurun, permintaan yang tetap atau meningkat dapat menyebabkan harga naik karena penawaran yang tidak mencukupi.
2. **Pasokan yang menurun**: Provinsi dengan pasokan terendah, Sulawesi Sel, memiliki pasokan yang relatif rendah sepanjang tahun. Hal ini dapat mempengaruhi keseluruhan pasokan beras dan menyebabkan harga naik.
3. **Fluktuasi musiman**: Harga beras dapat dipengaruhi oleh fluktuasi musiman, seperti musim tanam dan musim panen. Jika musim tanam atau musim panen tidak berjalan dengan baik, pasokan beras dapat menurun dan menyebabkan harga naik.
4. **Biaya produksi yang meningkat**: Biaya produksi, seperti biaya pupuk, biaya tenaga kerja, dan biaya lainnya, dapat meningkat dan mempengaruhi harga beras.
5. **Faktor eksternal**: Faktor eksternal seperti perubahan cuaca, bencana alam, atau perubahan kebij

In [5]:
print(copilot("Kapan stok terendah terjadi?"))

Stok terendah terjadi pada Oktober 2024, yaitu sebesar 2.900 ton di Sulawesi Sel.


In [6]:
print(copilot("Provinsi mana yang paling berisiko kekurangan stok?"))

Berdasarkan data yang ada, Sulawesi Selatan memiliki stok terkait dengan tanggal tertentu, terutama pada bulan Oktober 2024 dengan stok terendah sebesar 2900 ton. Selain itu, provinsi ini juga memiliki pasokan terendah. Oleh karena itu, provinsi Sulawesi Selatan paling berisiko kekurangan stok.


---
## 4. Konsep RAG

| Tanpa RAG | Dengan RAG |
|---|---|
| AI menjawab dari pengetahuan umum | AI menjawab dari data spesifik yang diberikan |
| Tidak akurat untuk data internal | Akurat dan relevan |
| Tidak bisa diperbarui real-time | Bisa diperbarui dengan data baru |

RAG adalah fondasi dari banyak aplikasi enterprise AI saat ini.